In [91]:
import json
import json
import re

In [2]:
class AXCompressor:
    """
    Advanced processor that prunes 'dead' branches, flattens useless wrappers, 
    and minimizes token usage for LLM consumption.
    """

    # Map long keys to short codes to save tokens
    KEY_MAP = {
        "role": "r",
        "description": "d",
        "value": "v", 
        "name": "n",
        "label": "l",
        "title": "t",
        "children": "ch",
        "text": "txt"
    }
    
    # Attributes that make a node 'semantically interesting'
    SEMANTIC_KEYS = {'description', 'value', 'name', 'label', 'title', 'text', 'hint'}

    @staticmethod
    def load_ax(ax_path) -> list[dict]:
        with open(ax_path, "r", encoding="utf-8") as file:
            return json.load(file)

    @staticmethod
    def is_visible(node: dict) -> bool:
        """
        Heuristic to check visibility/enabled status. 
        Checks both app-style and browser-style common keys.
        """
        # If 'visible' key exists and is False, it's hidden
        if node.get('visible') is False: 
            return False
        # If 'enabled' key exists and is False, it's disabled (optional: context dependent)
        # typically we want to know about disabled buttons, but not disabled hidden views.
        # Let's assume we want all enabled nodes or nodes with no status.
        return True

    @staticmethod
    def clean_and_compress(node: dict, is_browser: bool = False) -> dict | None:
        """
        Recursively cleans, flattens, and compresses the tree.
        Returns None if the node (and its subtree) contains no useful info.
        """
        
        # 1. Early Exit: Invisible nodes
        if not AXCompressor.is_visible(node):
            return None

        # 2. Extract Core Info
        # We normalize diverse keys into a standard set (role, desc, value)
        clean_node = {}
        
        # Capture Role
        role = node.get('role') or node.get('chromeRole') or node.get('role_description')
        if role:
            clean_node['r'] = role # Mapped to 'r'

        # Capture Semantic Content (Desc, Value, Name)
        # We concatenate them to save tokens if they are distinct, or just pick the best one.
        content = []
        for k in AXCompressor.SEMANTIC_KEYS:
            val = node.get(k)
            if val and isinstance(val, str) and val not in content:
                content.append(val)
        
        if content:
            # Join semantic parts, e.g., "Submit Button"
            clean_node['d'] = " | ".join(content) 

        # 3. Process Children (The Recursive Step)
        raw_children = node.get('children') or node.get('childIds') or []
        processed_children = []
        
        if isinstance(raw_children, list):
            for child in raw_children:
                # Handle browser implementation where childIds might be IDs, not dicts
                # (Assuming your input handles the resolution, otherwise we skip)
                if isinstance(child, dict):
                    result = AXCompressor.clean_and_compress(child, is_browser)
                    if result:
                        processed_children.append(result)

        # 4. FLATTENING STRATEGY (Tree Shaking)
        # If this node has NO content (no text/value) and...
        has_content = 'd' in clean_node
        
        # Case A: ...has no children -> It is a dead leaf. Kill it.
        if not has_content and not processed_children:
            return None

        # Case B: ...has exactly ONE child -> It is a useless wrapper. Replace self with child.
        # (Unless it has a specific meaningful role like 'button' or 'checkbox' that implies interaction)
        interactive_roles = ['button', 'checkbox', 'switch', 'link', 'input', 'textfield']
        is_interactive = any(r in str(clean_node.get('r', '')).lower() for r in interactive_roles)

        if not has_content and len(processed_children) == 1 and not is_interactive:
            return processed_children[0]

        # Case C: ...has children. Keep it, attach children.
        if processed_children:
            clean_node['ch'] = processed_children

        return clean_node

    @staticmethod
    def to_compact_string(node: dict, depth=0) -> str:
        """
        Converts the dict to a custom indentation-based string format.
        Much cheaper than JSON for LLMs.
        Format: [Role] "Description"
        """
        indent = "  " * depth
        role = node.get('r', 'Group')
        desc = f' "{node["d"]}"' if 'd' in node else ""
        
        line = f"{indent}[{role}]{desc}"
        
        lines = [line]
        for child in node.get('ch', []):
            lines.append(AXCompressor.to_compact_string(child, depth + 1))
            
        return "\n".join(lines)

    @staticmethod
    def process_ax(data: list[dict], output_format='json') -> list[dict]:
        final_output = []

        for i, row in enumerate(data):
            ax = row.get('accessibility')
            if not ax: continue
            
            raw_root = ax.get('data')
            if not raw_root: continue

            # Handle List vs Dict Root (Your VirtualRoot Logic)
            if isinstance(raw_root, list):
                if len(raw_root) == 1:
                    root_node = raw_root[0]
                else:
                    # Deduplicate
                    unique_nodes = []
                    seen = set()
                    for n in raw_root:
                        s = str(n)
                        if s not in seen:
                            unique_nodes.append(n)
                            seen.add(s)
                    root_node = {"role": "Root", "children": unique_nodes}
            elif isinstance(raw_root, dict):
                root_node = raw_root
            else:
                continue

            # --- THE PROCESSING ---
            is_browser = ax.get('is_browser', False)
            
            # 1. Clean and Tree Shake
            clean_root = AXCompressor.clean_and_compress(root_node, is_browser)
            
            if not clean_root:
                continue

            # 2. Format Output
            if output_format == 'string':
                # Returns a string representation directly
                payload = AXCompressor.to_compact_string(clean_root)
            else:
                # Returns the minified JSON
                payload = clean_root

            final_output.append({
                'id': i,
                'type': 'BROWSER' if is_browser else 'APP',
                'accessibility': payload
            })

        return final_output

In [59]:
class AXCompressor2:
    # ... [Mappings remain the same] ...
    KEY_MAP = {
        "role": "r", "description": "d", "value": "v", 
        "name": "n", "label": "l", "title": "t", 
        "children": "ch", "text": "txt"
    }
    
    SEMANTIC_KEYS = {'description', 'value', 'name', 'label', 'title', 'text', 'hint'}

    @staticmethod
    def load_ax(ax_path) -> list[dict]:
        with open(ax_path, "r", encoding="utf-8") as file:
            return json.load(file)

    @staticmethod
    def is_visible(node: dict) -> bool:
        if node.get('visible') is False: return False
        return True

    @staticmethod
    def clean_and_compress(node: dict, is_browser: bool = False) -> dict | None:
        # 1. Check Visibility
        if node.get('visible') is False: return None

        # 2. Extract Content
        clean_node = {}
        role = node.get('role') or node.get('chromeRole') or node.get('role_description')
        if role: clean_node['r'] = role

        # FIX 1: specific class reference updated
        content_parts = []
        for k in AXCompressor2.SEMANTIC_KEYS: 
            val = node.get(k)
            if val and isinstance(val, str) and val not in content_parts:
                content_parts.append(val)
        
        if content_parts:
            clean_node['d'] = " | ".join(content_parts)

        # 3. Process Children Recursively
        raw_children = node.get('children') or node.get('childIds') or []
        processed_children = []
        
        if isinstance(raw_children, list):
            for child in raw_children:
                if isinstance(child, dict):
                    # FIX 2: specific class reference updated
                    result = AXCompressor2.clean_and_compress(child, is_browser) 
                    if result:
                        processed_children.append(result)

        # --- REDUNDANCY FIX ---
        final_children = []
        parent_text = clean_node.get('d', "").strip() # Added .strip()
        
        for child in processed_children:
            child_text = child.get('d', "").strip() # Added .strip()
            
            # Logic: Match content, ensure child is a leaf
            is_redundant = (
                child_text and 
                parent_text and 
                child_text == parent_text and 
                'ch' not in child
            )
            
            if not is_redundant:
                final_children.append(child)
        
        processed_children = final_children
        # ----------------------

        # 4. FLATTENING
        has_content = 'd' in clean_node
        
        # Dead leaf?
        if not has_content and not processed_children:
            return None

        # Useless Wrapper? (Hoist)
        interactive_roles = ['button', 'checkbox', 'switch', 'link', 'input', 'textfield']
        current_role = str(clean_node.get('r', '')).lower()
        is_interactive = any(r in current_role for r in interactive_roles)

        if not has_content and len(processed_children) == 1 and not is_interactive:
            return processed_children[0]

        if processed_children:
            clean_node['ch'] = processed_children

        return clean_node

    @staticmethod
    def to_compact_string(node: dict, depth=0) -> str:
        indent = "  " * depth
        role = node.get('r', 'Group')
        desc = f' "{node["d"]}"' if 'd' in node else ""
        
        line = f"{indent}[{role}]{desc}"
        
        lines = [line]
        for child in node.get('ch', []):
            # FIX 3: specific class reference updated
            lines.append(AXCompressor2.to_compact_string(child, depth + 1)) 
            
        return "\n".join(lines)

    @staticmethod
    def process_ax(data: list[dict], output_format='json') -> list[dict]:
        final_output = []

        for i, row in enumerate(data):
            ax = row.get('accessibility')
            if not ax: continue
            
            raw_root = ax.get('data')
            if not raw_root: continue

            if isinstance(raw_root, list):
                if len(raw_root) == 1:
                    root_node = raw_root[0]
                else:
                    unique_nodes = []
                    seen = set()
                    for n in raw_root:
                        s = str(n)
                        if s not in seen:
                            unique_nodes.append(n)
                            seen.add(s)
                    root_node = {"role": "Root", "children": unique_nodes}
            elif isinstance(raw_root, dict):
                root_node = raw_root
            else:
                continue

            is_browser = ax.get('is_browser', False)
            
            # FIX 4: specific class reference updated
            clean_root = AXCompressor2.clean_and_compress(root_node, is_browser) 
            
            if not clean_root:
                continue

            if output_format == 'string':
                # FIX 5: specific class reference updated
                payload = AXCompressor2.to_compact_string(clean_root) 
            else:
                payload = clean_root

            final_output.append({
                'id': i,
                'type': 'BROWSER' if is_browser else 'APP',
                'accessibility': payload
            })

        return final_output

In [60]:
class AXCompressor3:
    """
    Fixed version that correctly handles:
    1. APP (Standard AX)
    2. BROWSER (Web/CDP)
    3. PDF (Summary text)
    """

    KEY_MAP = {
        "role": "r", "description": "d", "value": "v", 
        "name": "n", "label": "l", "title": "t", 
        "children": "ch", "text": "txt"
    }
    
    # Expanded keys to catch browser-specific text fields
    SEMANTIC_KEYS = {
        'description', 'value', 'name', 'label', 'title', 'text', 'hint', 
        'attributes' # Sometimes browser nodes hide info here
    }

    @staticmethod
    def load_ax(ax_path) -> list[dict]:
        with open(ax_path, "r", encoding="utf-8") as file:
            return json.load(file)

    @staticmethod
    def clean_and_compress(node: dict, is_browser: bool = False, is_root: bool = False) -> dict | None:
        """
        is_root=True ensures we don't return None for the top-level element 
        just because it lacks a description, provided it has children.
        """
        
        # 1. Check Visibility (Skip if strictly hidden)
        if node.get('visible') is False: 
            return None

        # 2. Extract Content
        clean_node = {}
        
        # Get Role
        role = node.get('role') or node.get('chromeRole') or node.get('role_description')
        if role: 
            clean_node['r'] = role

        # Get Text/Description
        content_parts = []
        for k in AXCompressor3.SEMANTIC_KEYS:
            val = node.get(k)
            # Handle standard string values
            if val and isinstance(val, str) and val not in content_parts:
                content_parts.append(val)
            # Handle Dictionary values (sometimes attributes are dicts)
            elif isinstance(val, dict):
                 for sub_k, sub_v in val.items():
                     if isinstance(sub_v, str) and sub_v not in content_parts:
                         content_parts.append(f"{sub_k}:{sub_v}")

        if content_parts:
            clean_node['d'] = " | ".join(content_parts)

        # 3. Process Children
        # Handle both App 'children' and Browser 'childIds' (assuming resolved or nested)
        raw_children = node.get('children') or node.get('childIds') or []
        processed_children = []
        
        if isinstance(raw_children, list):
            for child in raw_children:
                if isinstance(child, dict):
                    # Recursive call
                    result = AXCompressor3.clean_and_compress(child, is_browser, is_root=False)
                    if result:
                        processed_children.append(result)

        # --- REDUNDANCY FIX ---
        final_children = []
        parent_text = clean_node.get('d', "").strip()
        
        for child in processed_children:
            child_text = child.get('d', "").strip()
            # If child echoes parent and is a leaf, drop it
            is_redundant = (
                child_text and parent_text and 
                child_text == parent_text and 
                'ch' not in child
            )
            if not is_redundant:
                final_children.append(child)
        
        processed_children = final_children
        # ----------------------

        # 4. FLATTENING & HOISTING
        has_content = 'd' in clean_node
        
        # Case A: Dead Leaf (No content, no children)
        if not has_content and not processed_children:
            # If it's the Root, we might return an empty object just to show it exists,
            # but usually, an empty tree is useless.
            return None

        # Case B: Useless Wrapper (Hoist)
        # We DO NOT hoist if this is the Root (we want to keep the top container)
        # We DO NOT hoist if it's interactive (buttons, inputs)
        interactive_roles = ['button', 'checkbox', 'switch', 'link', 'input', 'textfield']
        current_role = str(clean_node.get('r', '')).lower()
        is_interactive = any(r in current_role for r in interactive_roles)

        if not is_root and not has_content and len(processed_children) == 1 and not is_interactive:
            return processed_children[0]

        # Attach children if they exist
        if processed_children:
            clean_node['ch'] = processed_children
        elif is_root and not has_content and not 'r' in clean_node:
             # If root is totally empty, return None
             return None

        return clean_node

    @staticmethod
    def to_compact_string(node: dict, depth=0) -> str:
        indent = "  " * depth
        role = node.get('r', 'Group')
        desc = f' "{node["d"]}"' if 'd' in node else ""
        
        line = f"{indent}[{role}]{desc}"
        
        lines = [line]
        for child in node.get('ch', []):
            lines.append(AXCompressor3.to_compact_string(child, depth + 1))
            
        return "\n".join(lines)

    @staticmethod
    def process_ax(data: list[dict], output_format='json') -> list[dict]:
        final_output = []

        for i, row in enumerate(data):
            ax = row.get('accessibility')
            if not ax: continue
            
            raw_root = ax.get('data')
            if not raw_root: continue
            
            is_browser = ax.get('is_browser', False)

            # --- 1. HANDLE ROOT STRUCTURE ---
            if isinstance(raw_root, list):
                if not raw_root: continue
                if len(raw_root) == 1:
                    root_node = raw_root[0]
                else:
                    # Deduplicate multiple windows
                    unique_nodes = []
                    seen = set()
                    for n in raw_root:
                        s = str(n)
                        if s not in seen:
                            unique_nodes.append(n)
                            seen.add(s)
                    root_node = {"role": "VirtualRoot", "children": unique_nodes}
            elif isinstance(raw_root, dict):
                root_node = raw_root
            else:
                continue

            # --- 2. HANDLE PDF vs STANDARD TREE ---
            
            # CASE: PDF
            if root_node.get('is_pdf'):
                # Extract summary explicitly as the original code did
                pdf_summary = root_node.get('pdf_summary', {})
                # Normalize it for our output format
                clean_root = {
                    "r": "PDF Document",
                    "d": str(pdf_summary) if isinstance(pdf_summary, str) else str(pdf_summary.get('text', pdf_summary))
                }
                type_label = "PDF"

            # CASE: STANDARD APP / BROWSER
            else:
                # Pass is_root=True so we don't accidentally delete the main window 
                # if it has no description (common in Browsers)
                clean_root = AXCompressor3.clean_and_compress(root_node, is_browser, is_root=True)
                type_label = 'BROWSER' if is_browser else 'APP'
            
            if not clean_root:
                continue

            # --- 3. FORMAT OUTPUT ---
            if output_format == 'string':
                payload = AXCompressor3.to_compact_string(clean_root)
            else:
                payload = clean_root

            final_output.append({
                'id': i,
                'type': type_label,
                'accessibility': payload
            })

        return final_output

In [ ]:
# Example Usage
processor = AXCompressor()
data = processor.load_ax("../input_data/max/accessibility.json")

In [9]:
print(f"RAW {len(data)} records.")
len(str(data).split(' '))

RAW 69 records.


477287

In [10]:
# Option A: Get Minified JSON (Good for data storage)
clean_json = processor.process_ax(data, output_format='json')

# Check reduction
print(f"Processed {len(clean_json)} records.")
len(str(clean_json).split(' '))

Processed 43 records.


39658

In [18]:
clean_json[0]['accessibility']

{'r': 'AXWindow',
 'd': 'context-user-model_–_image-1.png',
 'ch': [{'r': 'AXStaticText', 'd': 'context-user-model – image-1.png'},
  {'r': 'AXGroup',
   'd': 'Root Pane',
   'ch': [{'r': 'AXGroup',
     'd': 'Status Bar',
     'ch': [{'r': 'AXStaticText', 'd': 'Make file read-only'},
      {'r': 'AXStaticText', 'd': '<No interpreter>'},
      {'r': 'AXGroup',
       'd': 'Navigation Bar',
       'ch': [{'r': 'AXButton', 'd': 'context-user-model'}]},
      {'r': 'AXStaticText', 'd': 'Column selection mode'},
      {'r': 'AXStaticText', 'd': 'Go to Line'},
      {'r': 'AXLink', 'd': 'Show all (2)'}]},
    {'r': 'AXGroup',
     'd': 'Local Tool Window',
     'ch': [{'r': 'AXGroup',
       'd': 'Action Toolbar',
       'ch': [{'r': 'AXButton', 'd': 'Hide'},
        {'r': 'AXButton', 'd': 'Options'}]},
      {'r': 'AXGroup',
       'd': 'Action Toolbar',
       'ch': [{'r': 'AXButton', 'd': 'New Predefined Session'},
        {'r': 'AXButton', 'd': 'New Tab'}]},
      {'r': 'AXStaticText', 

In [ ]:
# Option B: Get Tree String (BEST for LLM Context Window)
clean_str = processor.process_ax(data, output_format='string')

# Check reduction
print(f"Processed {len(clean_json)} records.")
len(str(clean_json).split(' '))

Processed 43 records.


39658

In [14]:
# Check reduction
print(f"Processed {len(clean_str)} records.")
if len(clean_str) > 0:
    print("--- Sample Record (Tree String Format) ---")
    print(clean_str[0]['accessibility'])

Processed 43 records.
--- Sample Record (Tree String Format) ---
[AXWindow] "context-user-model_–_image-1.png"
  [AXStaticText] "context-user-model – image-1.png"
  [AXGroup] "Root Pane"
    [AXGroup] "Status Bar"
      [AXStaticText] "Make file read-only"
      [AXStaticText] "<No interpreter>"
      [AXGroup] "Navigation Bar"
        [AXButton] "context-user-model"
      [AXStaticText] "Column selection mode"
      [AXStaticText] "Go to Line"
      [AXLink] "Show all (2)"
    [AXGroup] "Local Tool Window"
      [AXGroup] "Action Toolbar"
        [AXButton] "Hide"
        [AXButton] "Options"
      [AXGroup] "Action Toolbar"
        [AXButton] "New Predefined Session"
        [AXButton] "New Tab"
      [AXStaticText] "Local"
      [AXStaticText] "Terminal"
    [AXGroup] "Right Toolbar"
      [AXButton] "Database"
      [AXButton] "AI Assistant"
      [AXButton] "Notifications"
      [AXButton] "More"
    [AXTabGroup] "{
    "id": "",
    "name": null,
    "role": "No role",
    "descr

## Parent-Child Echo

In [61]:
# Example Usage
processor = AXCompressor3()
data = processor.load_ax("../input_data/accessibility_marichka.json")

In [64]:
print(f"RAW {len(data)} records.")
len(str(data).split(' '))

RAW 350 records.


7087621

In [65]:
# Option A: Get Minified JSON (Good for data storage)
clean_json = processor.process_ax(data, output_format='json')

# Check reduction
print(f"Processed {len(clean_json)} records.")
len(str(clean_json).split(' '))

Processed 74 records.


71475

In [66]:
# Option B: Get Tree String (BEST for LLM Context Window)
clean_str = processor.process_ax(data, output_format='string')

# Check reduction
print(f"Processed {len(clean_json)} records.")
len(str(clean_json).split(' '))

Processed 74 records.


71475

In [67]:
# Check reduction
print(f"Processed {len(clean_str)} records.")
if len(clean_str) > 0:
    print("--- Sample Record (Tree String Format) ---")
    print(clean_str[8]['accessibility'])

Processed 74 records.
--- Sample Record (Tree String Format) ---
[PDF Document] "[Error extracting PDF text: partial fetchs failed or PDF is image/scanned]"


In [126]:
clean_str[12]

{'id': 61,
 'type': 'PDF',
 'accessibility': '[PDF Document] "[Error extracting PDF text: partial fetchs failed or PDF is image/scanned]"'}

# Block-dwon approach

In [69]:
class AXCompressor4:
    """
    Robust version. Uses a 'Blocklist' strategy instead of 'Allowlist'.
    Keeps any string data found in the node unless it is explicitly ignored (e.g., IDs, coords).
    """

    KEY_MAP = {
        "role": "r", "description": "d", "value": "v", 
        "name": "n", "label": "l", "title": "t", 
        "children": "ch", "text": "txt"
    }

    # Keys that we KNOW are not useful for the LLM. 
    # We strip these. We keep EVERYTHING else.
    IGNORED_KEYS = {
        # ID / Tech junk
        'id', 'nodeId', 'backendDOMNodeId', 'backendNodeId', 'parentId', 'frameId', 'childIds',
        'idx', 'index', 'pid', 'windowId',
        # Coordinates / Visuals
        'bbox', 'rect', 'position', 'size', 'absolute_position', 'visible_bbox', 
        'x', 'y', 'width', 'height', 'top', 'left', 'bottom', 'right',
        # Status flags (unless you specifically need them)
        'visible', 'enabled', 'focused', 'selected', 'checked', 'busy',
        # Structure
        'children', 'nodes', 'parent', 'superseded'
    }

    @staticmethod
    def load_ax(ax_path) -> list[dict]:
        with open(ax_path, "r", encoding="utf-8") as file:
            return json.load(file)

    @staticmethod
    def extract_content_smart(node: dict) -> list[str]:
        """
        Iterates over ALL keys. Keeps any string/number that isn't in IGNORED_KEYS.
        """
        content = []
        
        # 1. Prioritize Main Role (if exists)
        role = node.get('role') or node.get('chromeRole') or node.get('role_description')
        
        # 2. Hunt for any other text
        for key, val in node.items():
            if key in AXCompressor4.IGNORED_KEYS:
                continue
            
            # Skip the role key itself if we already grabbed it
            if key in ['role', 'chromeRole', 'role_description']:
                continue

            if val is not None:
                # If string and not empty
                if isinstance(val, str) and val.strip():
                    if val not in content: # Avoid duplicates
                        content.append(val)
                # If number (sometimes useful, e.g. slider values)
                elif isinstance(val, (int, float)) and not isinstance(val, bool):
                    content.append(str(val))
                # If dict (e.g. attributes), try to stringify
                elif isinstance(val, dict):
                     # Naive extraction of values from simple dicts
                     for k, v in val.items():
                         if isinstance(v, str) and v.strip() and v not in content:
                             content.append(v)

        return role, content

    @staticmethod
    def clean_and_compress(node: dict, is_browser: bool = False, is_root: bool = False) -> dict | None:
        
        # 1. Visibility Check (Safer: default to True if missing)
        if node.get('visible') is False: 
            return None

        # 2. Smart Content Extraction
        role, content_list = AXCompressor4.extract_content_smart(node)
        
        clean_node = {}
        if role: 
            clean_node['r'] = role
        if content_list:
            clean_node['d'] = " | ".join(content_list)

        # 3. Find Children (Check all common keys)
        raw_children = node.get('children') or node.get('childIds') or node.get('nodes') or []
        processed_children = []
        
        if isinstance(raw_children, list):
            for child in raw_children:
                if isinstance(child, dict):
                    result = AXCompressor4.clean_and_compress(child, is_browser, is_root=False)
                    if result:
                        processed_children.append(result)

        # --- REDUNDANCY FIX (Parent vs Child Echo) ---
        final_children = []
        parent_text = clean_node.get('d', "").strip()
        
        for child in processed_children:
            child_text = child.get('d', "").strip()
            # If child is a leaf AND echoes parent text exactly -> Drop it
            is_redundant = (
                child_text and parent_text and 
                child_text == parent_text and 
                'ch' not in child
            )
            if not is_redundant:
                final_children.append(child)
        
        processed_children = final_children
        # ---------------------------------------------

        # 4. HOISTING / FLATTENING
        has_content = 'd' in clean_node
        has_role = 'r' in clean_node

        # Case A: Dead Leaf (No content, no children)
        if not has_content and not processed_children:
            # If it's a Root with no content/children, it's useless (unless it has a Role)
            # We keep it only if it's the root so the record exists.
            if is_root and has_role:
                return clean_node
            return None

        # Case B: Useless Wrapper
        # If no text content, and exactly 1 child -> replace self with child
        # EXCEPTION: If is_root, never hoist (we want the container)
        # EXCEPTION: If it is an interactive element (button), keep it even if wrapper
        interactive_roles = ['button', 'checkbox', 'switch', 'link', 'input', 'textfield', 'a', 'area']
        current_role = str(clean_node.get('r', '')).lower()
        is_interactive = any(r in current_role for r in interactive_roles)

        if not is_root and not has_content and len(processed_children) == 1 and not is_interactive:
            return processed_children[0]

        # Attach Children
        if processed_children:
            clean_node['ch'] = processed_children

        return clean_node

    @staticmethod
    def to_compact_string(node: dict, depth=0) -> str:
        indent = "  " * depth
        role = node.get('r', 'Group')
        desc = f' "{node["d"]}"' if 'd' in node else ""
        
        line = f"{indent}[{role}]{desc}"
        
        lines = [line]
        for child in node.get('ch', []):
            lines.append(AXCompressor4.to_compact_string(child, depth + 1))
            
        return "\n".join(lines)

    @staticmethod
    def process_ax(data: list[dict], output_format='json') -> list[dict]:
        final_output = []

        for i, row in enumerate(data):
            ax = row.get('accessibility')
            if not ax: continue
            
            raw_root = ax.get('data')
            if not raw_root: continue
            
            is_browser = ax.get('is_browser', False)

            # --- ROOT NORMALIZATION ---
            if isinstance(raw_root, list):
                if not raw_root: continue
                if len(raw_root) == 1:
                    root_node = raw_root[0]
                else:
                    # Deduplicate Logic
                    unique_nodes = []
                    seen = set()
                    for n in raw_root:
                        s = str(n)
                        if s not in seen:
                            unique_nodes.append(n)
                            seen.add(s)
                    root_node = {"role": "VirtualRoot", "children": unique_nodes}
            elif isinstance(raw_root, dict):
                root_node = raw_root
            else:
                continue

            # --- PROCESS ---
            if root_node.get('is_pdf'):
                # PDF Handling
                pdf_summary = root_node.get('pdf_summary', {})
                text = str(pdf_summary) if isinstance(pdf_summary, str) else str(pdf_summary.get('text', pdf_summary))
                clean_root = {"r": "PDF", "d": text}
                type_label = "PDF"
            else:
                # App/Browser Handling
                # We pass is_root=True to prevent the top-level window from disappearing
                clean_root = AXCompressor4.clean_and_compress(root_node, is_browser, is_root=True)
                type_label = 'BROWSER' if is_browser else 'APP'
            
            if not clean_root:
                # Fallback: if everything was filtered out, maybe it's just raw text?
                continue

            # --- FORMAT ---
            if output_format == 'string':
                payload = AXCompressor4.to_compact_string(clean_root)
            else:
                payload = clean_root

            final_output.append({
                'id': i,
                'type': type_label,
                'accessibility': payload
            })

        return final_output

In [71]:

# Point this to your file
data = AXCompressor4.load_ax("../input_data/accessibility_marichka.json") 

# 1. Test String Format (Best for LLM)
processed = AXCompressor4.process_ax(data, output_format='string')
print(f"Processed {len(processed)} / {len(data)} records.")

Processed 346 / 350 records.


In [74]:
# Print the first few types to ensure we have Browsers and Apps
types = [p['type'] for p in processed[:10]]
print(f"Types found in first 10: {types}")

Types found in first 10: ['APP', 'APP', 'APP', 'BROWSER', 'BROWSER', 'BROWSER', 'BROWSER', 'BROWSER', 'BROWSER', 'BROWSER']


In [89]:
processed[0]

{'id': 0,
 'type': 'APP',
 'accessibility': '[AXWindow] "gusik_(DM)_-_macpaw_-_2_new_items_-_Slack | Slack"\n  [AXGroup] "gusik_(DM)_-_macpaw_-_2_new_items_-_Slack"'}

In [80]:
print(processed[4]['accessibility'])

[Group]
  [{'type': 'internalRole', 'value': 'RootWebArea'}] "computedString | xlangai/AgentNet · Datasets at Hugging Face"
  [{'type': 'role', 'value': 'banner'}] "computedString"
  [{'type': 'role', 'value': 'main'}] "computedString"
  [{'type': 'role', 'value': 'contentinfo'}] "computedString"
  [{'type': 'role', 'value': 'link'}] "computedString | Hugging Face's logo Hugging Face"
  [{'type': 'role', 'value': 'generic'}] "computedString"
  [{'type': 'role', 'value': 'navigation'}] "computedString | Main"
  [{'type': 'role', 'value': 'sectionheader'}] "computedString"
  [{'type': 'role', 'value': 'generic'}] "computedString"
  [{'type': 'role', 'value': 'navigation'}] "computedString"
  [{'type': 'role', 'value': 'image'}] "computedString | Hugging Face's logo"
  [{'type': 'internalRole', 'value': 'StaticText'}] "computedString | Hugging Face"
  [{'type': 'role', 'value': 'textbox'}] "computedString | Search models, datasets, users..."
  [{'type': 'role', 'value': 'list'}] "computed

In [82]:
len(str(processed).split(' '))

1093056

In [83]:
len(processed)

346

In [90]:
[(len(str(record).split(' ')), record['type']) for record in processed]

[(12, 'APP'),
 (12, 'APP'),
 (12, 'APP'),
 (57217, 'BROWSER'),
 (57217, 'BROWSER'),
 (57217, 'BROWSER'),
 (12, 'BROWSER'),
 (12, 'BROWSER'),
 (12, 'BROWSER'),
 (12, 'BROWSER'),
 (12, 'BROWSER'),
 (12, 'APP'),
 (12, 'APP'),
 (12, 'APP'),
 (1047, 'BROWSER'),
 (1047, 'BROWSER'),
 (1047, 'BROWSER'),
 (1397, 'APP'),
 (1368, 'APP'),
 (1047, 'BROWSER'),
 (12, 'BROWSER'),
 (12, 'BROWSER'),
 (12, 'BROWSER'),
 (1047, 'BROWSER'),
 (1047, 'BROWSER'),
 (12, 'BROWSER'),
 (186, 'BROWSER'),
 (186, 'BROWSER'),
 (186, 'BROWSER'),
 (186, 'BROWSER'),
 (186, 'BROWSER'),
 (186, 'BROWSER'),
 (186, 'BROWSER'),
 (1047, 'BROWSER'),
 (12, 'BROWSER'),
 (12, 'BROWSER'),
 (12, 'BROWSER'),
 (12, 'BROWSER'),
 (186, 'BROWSER'),
 (186, 'BROWSER'),
 (186, 'BROWSER'),
 (12, 'BROWSER'),
 (12, 'BROWSER'),
 (1047, 'BROWSER'),
 (1047, 'BROWSER'),
 (1047, 'BROWSER'),
 (1047, 'BROWSER'),
 (1047, 'BROWSER'),
 (186, 'BROWSER'),
 (186, 'BROWSER'),
 (1825, 'BROWSER'),
 (1825, 'BROWSER'),
 (1825, 'BROWSER'),
 (17, 'PDF'),
 (17, 'PD

# Validation

In [94]:
class AXValidator:
    """
    Compares Raw Accessibility Data vs. Compressed Output.
    Goal: Ensure semantic text is preserved while technical garbage is discarded.
    """

    SEMANTIC_KEYS = {'value', 'name', 'label', 'title', 'description', 'text', 'hint', 'role'}
    # Heuristic: Ignore tokens that look like UUIDs, hex codes, or coordinates
    GARBAGE_REGEX = re.compile(r'^(?:[0-9]+|0x[0-9a-f]+|[0-9a-f]{8,}|true|false|null|undefined)$', re.IGNORECASE)

    @staticmethod
    def tokenize(text: str) -> set[str]:
        if not text: return set()
        tokens = re.split(r'[^a-zA-Z0-9]+', str(text).lower())
        return {t for t in tokens if t and len(t) > 1}

    @staticmethod
    def extract_tokens_from_raw(node: dict | list) -> set[str]:
        tokens = set()
        if isinstance(node, list):
            for item in node:
                tokens.update(AXValidator.extract_tokens_from_raw(item))
            return tokens
        
        if isinstance(node, dict):
            for k, v in node.items():
                if k in AXValidator.SEMANTIC_KEYS and isinstance(v, (str, int)):
                    tokens.update(AXValidator.tokenize(v))
                if isinstance(v, (dict, list)):
                    tokens.update(AXValidator.extract_tokens_from_raw(v))
        return tokens

    @staticmethod
    def extract_tokens_from_compressed(compressed_data) -> set[str]:
        if isinstance(compressed_data, (dict, list)):
            return AXValidator.extract_tokens_from_raw(compressed_data)
        elif isinstance(compressed_data, str):
            return AXValidator.tokenize(compressed_data)
        return set()

    @staticmethod
    def compare_record(raw_record: dict, compressed_record: dict, index: int):
        """
        Calculates token overlap.
        """
        raw_root = raw_record.get('accessibility', {}).get('data')
        comp_payload = compressed_record.get('accessibility')

        if not raw_root or not comp_payload:
            return None

        raw_tokens = AXValidator.extract_tokens_from_raw(raw_root)
        comp_tokens = AXValidator.extract_tokens_from_compressed(comp_payload)

        # Filter garbage from Raw to avoid false negatives
        meaningful_raw = {t for t in raw_tokens if not AXValidator.GARBAGE_REGEX.match(t)}
        
        missing_tokens = meaningful_raw - comp_tokens
        
        if len(meaningful_raw) == 0:
            score = 100.0
        else:
            score = 100.0 * (1 - (len(missing_tokens) / len(meaningful_raw)))

        # FIX: Use index if ID is missing in raw data
        record_id = raw_record.get('id', index)

        return {
            "id": record_id,
            "score": round(score, 2),
            "raw_count": len(meaningful_raw),
            "comp_count": len(comp_tokens),
            "missing_sample": list(missing_tokens)[:10]
        }

    @staticmethod
    def validate_dataset(raw_path, compressed_output):
        with open(raw_path, 'r', encoding='utf-8') as f:
            raw_data = json.load(f)

        print(f"{'ID':<5} | {'Score':<6} | {'Status':<10} | {'Missing Words Sample'}")
        print("-" * 100)

        scores = []
        for i, raw_row in enumerate(raw_data):
            if i >= len(compressed_output): break
            
            comp_row = compressed_output[i]
            
            # Pass 'i' as the index fallback
            report = AXValidator.compare_record(raw_row, comp_row, i)
            if not report: continue

            scores.append(report['score'])
            
            status = "✅ OK" if report['score'] > 85 else "⚠️ LOW"
            if report['score'] < 50: status = "❌ BAD"
            
            missing_str = ", ".join(report['missing_sample'])
            
            # FIX: Convert ID to str to avoid format errors if it's somehow weird
            r_id = str(report['id'])
            print(f"{r_id:<5} | {report['score']}% | {status:<10} | {missing_str}")

        avg = sum(scores)/len(scores) if scores else 0
        print("-" * 100)
        print(f"Average Semantic Preservation: {avg:.2f}%")

In [95]:

raw_path = "../input_data/accessibility_marichka.json"
raw_data = AXCompressor4.load_ax(raw_path)
compressed_data = AXCompressor4.process_ax(raw_data, output_format='string')

# 2. Run the Validator
AXValidator.validate_dataset(raw_path, compressed_data)

ID    | Score  | Status     | Missing Words Sample
----------------------------------------------------------------------------------------------------
0     | 88.89% | ✅ OK       | axbutton
1     | 88.89% | ✅ OK       | axbutton
2     | 90.0% | ✅ OK       | axbutton
4     | 97.77% | ✅ OK       | join, workable, plaintext, editable, notrendered, login, none, f0, emptyalt, evnpqcek9qshpqzx
5     | 97.77% | ✅ OK       | join, workable, plaintext, editable, notrendered, login, none, f0, emptyalt, evnpqcek9qshpqzx
6     | 0.21% | ❌ BAD      | bd68, plaintext, distinctive, deleted, burn, bca8, pagination, underline, licensed, right
7     | 12.5% | ❌ BAD      | focused, newtab, focusable, uninteresting, none, url, about
8     | 12.5% | ❌ BAD      | focused, newtab, focusable, uninteresting, none, url, about
9     | 12.5% | ❌ BAD      | focused, newtab, focusable, uninteresting, none, url, about
10    | 12.5% | ❌ BAD      | focused, newtab, focusable, uninteresting, none, url, about
11    | 0

In [96]:
class AXCompressor5:
    """
    The 'Deep Search' version. 
    It recursively hunts for text in nested dictionaries (attributes, properties) 
    to ensure no semantic data is left behind.
    """

    KEY_MAP = {
        "role": "r", "description": "d", "value": "v", 
        "name": "n", "label": "l", "title": "t", 
        "children": "ch", "text": "txt"
    }

    # STRICT Garbage list. We ignore these, but we look at EVERYTHING else.
    IGNORED_KEYS = {
        # IDs and Tech references
        'id', 'nodeId', 'backendDOMNodeId', 'backendNodeId', 'parentId', 'frameId', 'childIds',
        'idx', 'index', 'pid', 'windowId', 'relation', 'relations',
        # Geometry / Visuals
        'bbox', 'rect', 'position', 'size', 'absolute_position', 'visible_bbox', 
        'x', 'y', 'width', 'height', 'top', 'left', 'bottom', 'right', 'clip',
        # Visual styling that usually doesn't add semantic meaning
        'color', 'backgroundColor', 'style', 'textStyle', 'layout',
        # Redundant structure
        'children', 'nodes', 'parent', 'superseded', 'is_pdf'
    }

    # Keys that specifically denote the "Type" of the element
    ROLE_KEYS = {'role', 'chromeRole', 'role_description', 'subrole', 'axRole', 'AXRole'}

    @staticmethod
    def load_ax(ax_path) -> list[dict]:
        with open(ax_path, "r", encoding="utf-8") as file:
            return json.load(file)

    @staticmethod
    def _is_garbage(text):
        """Returns True if text is likely tech garbage (hex codes, single chars)."""
        if not isinstance(text, str): return True
        s = text.strip()
        if len(s) < 2: return True # Skip single letters (usually)
        if s.lower() in {'true', 'false', 'none', 'null', 'undefined'}: return True
        return False

    @staticmethod
    def extract_deep_content(node: dict) -> tuple[str, list[str]]:
        """
        Recursively digs into a node (excluding children) to find ALL text.
        Handles: {"attributes": {"name": "foo"}} -> "name: foo"
        """
        role = None
        content = []

        # Helper to process values
        def add_val(k, v):
            if v is None: return
            
            # If it's a Role
            if k in AXCompressor5.ROLE_KEYS:
                nonlocal role
                # Prioritize human readable roles
                if not role or (len(v) > len(role) and 'AX' not in v): 
                    role = v
                return

            # If it's a Dictionary (e.g. 'attributes'), recurse!
            if isinstance(v, dict):
                for sub_k, sub_v in v.items():
                    if sub_k not in AXCompressor5.IGNORED_KEYS:
                        add_val(sub_k, sub_v)
                return

            # If it's a List (e.g. 'values': ['a', 'b']), join them
            if isinstance(v, list):
                for item in v:
                    if isinstance(item, (str, int, float)):
                        add_val(k, item)
                return

            # If it's a String/Number
            if isinstance(v, (str, int, float)) and not isinstance(v, bool):
                s_val = str(v).strip()
                if not AXCompressor5._is_garbage(s_val):
                    # Heuristic: If the key is meaningful (like 'url'), include it
                    if k in {'url', 'href', 'src'}:
                        if s_val not in content: content.append(f"{k}:{s_val}")
                    elif s_val not in content:
                        content.append(s_val)

        # Main Loop
        for key, val in node.items():
            if key in AXCompressor5.IGNORED_KEYS: continue
            add_val(key, val)

        return role, content

    @staticmethod
    def clean_and_compress(node: dict, is_browser: bool = False, is_root: bool = False) -> dict | None:
        
        # 1. Visibility Check (Skip explicitly hidden)
        # Note: Some trees define 'hidden' instead of 'visible'. Check both.
        if node.get('visible') is False: return None
        if node.get('hidden') is True: return None

        # 2. Deep Content Extraction
        role, content_list = AXCompressor5.extract_deep_content(node)
        
        clean_node = {}
        if role: 
            clean_node['r'] = role
        
        # If we found content, join it. 
        # We perform a quick dedupe in case 'value' and 'description' are identical.
        if content_list:
            clean_node['d'] = " | ".join(dict.fromkeys(content_list)) # preserves order, removes dupes

        # 3. Process Children
        # Look for every possible children key
        raw_children = (node.get('children') or 
                        node.get('childIds') or 
                        node.get('nodes') or 
                        node.get('AXChildren') or [])
        
        processed_children = []
        if isinstance(raw_children, list):
            for child in raw_children:
                if isinstance(child, dict):
                    result = AXCompressor5.clean_and_compress(child, is_browser, is_root=False)
                    if result:
                        processed_children.append(result)

        # --- REDUNDANCY FIX (Parent vs Child Echo) ---
        final_children = []
        parent_text = clean_node.get('d', "").strip()
        
        for child in processed_children:
            child_text = child.get('d', "").strip()
            # If child text is identical to parent text, AND child has no children of its own
            is_redundant = (
                child_text and parent_text and 
                child_text == parent_text and 
                'ch' not in child
            )
            if not is_redundant:
                final_children.append(child)
        processed_children = final_children
        # ---------------------------------------------

        # 4. HOISTING / FLATTENING
        has_content = 'd' in clean_node
        has_role = 'r' in clean_node

        # Case A: Dead Leaf (No content, no children)
        if not has_content and not processed_children:
            # If it's root, we keep it if it at least has a Role (e.g. "Window")
            if is_root and has_role: return clean_node
            return None

        # Case B: Useless Wrapper
        # Never hoist Root.
        # Never hoist Interactive elements (buttons need to exist even if they wrap text)
        interactive_roles = ['button', 'checkbox', 'switch', 'link', 'input', 'textfield', 'a', 'area', 'search']
        current_role = str(clean_node.get('r', '')).lower()
        is_interactive = any(r in current_role for r in interactive_roles)

        if not is_root and not has_content and len(processed_children) == 1 and not is_interactive:
            return processed_children[0]

        # Attach Children
        if processed_children:
            clean_node['ch'] = processed_children

        return clean_node

    @staticmethod
    def to_compact_string(node: dict, depth=0) -> str:
        indent = "  " * depth
        role = node.get('r', 'Group')
        desc = f' "{node["d"]}"' if 'd' in node else ""
        
        line = f"{indent}[{role}]{desc}"
        
        lines = [line]
        for child in node.get('ch', []):
            lines.append(AXCompressor5.to_compact_string(child, depth + 1))
            
        return "\n".join(lines)

    @staticmethod
    def process_ax(data: list[dict], output_format='json') -> list[dict]:
        final_output = []

        for i, row in enumerate(data):
            ax = row.get('accessibility')
            if not ax: continue
            
            raw_root = ax.get('data')
            if not raw_root: continue
            
            is_browser = ax.get('is_browser', False)

            # --- ROOT NORMALIZATION ---
            if isinstance(raw_root, list):
                if not raw_root: continue
                if len(raw_root) == 1:
                    root_node = raw_root[0]
                else:
                    # Deduplicate Logic
                    unique_nodes = []
                    seen = set()
                    for n in raw_root:
                        s = str(n)
                        if s not in seen:
                            unique_nodes.append(n)
                            seen.add(s)
                    root_node = {"role": "VirtualRoot", "children": unique_nodes}
            elif isinstance(raw_root, dict):
                root_node = raw_root
            else:
                continue

            # --- PDF HANDLING ---
            if root_node.get('is_pdf'):
                pdf_summary = root_node.get('pdf_summary', {})
                text = str(pdf_summary) if isinstance(pdf_summary, str) else str(pdf_summary.get('text', pdf_summary))
                # Only keep first 1000 chars of PDF to avoid blowing context
                clean_root = {"r": "PDF", "d": text[:1000] + "..." if len(text) > 1000 else text}
                type_label = "PDF"

            # --- STANDARD APP/BROWSER HANDLING ---
            else:
                clean_root = AXCompressor5.clean_and_compress(root_node, is_browser, is_root=True)
                type_label = 'BROWSER' if is_browser else 'APP'
            
            # --- FINAL FALLBACK ---
            # If clean_root is None (rare now), it means the tree was totally empty/invisible.
            if not clean_root:
                continue

            # --- FORMAT ---
            if output_format == 'string':
                payload = AXCompressor5.to_compact_string(clean_root)
            else:
                payload = clean_root

            final_output.append({
                'id': i,
                'type': type_label,
                'accessibility': payload
            })

        return final_output

In [108]:
class AXCompressor8:
    """
    The 'Alignment-Safe' Version.
    1. Never drops a record (preserves ID alignment).
    2. 'Vacuum' strategy: recursive text extraction for unknown structures.
    3. Preserves Interactive & Layout roles.
    """

    # BLOCKLIST: We ignore these. Everything else is treated as potential content.
    IGNORED_KEYS = {
        'id', 'nodeId', 'backendDOMNodeId', 'backendNodeId', 'parentId', 'frameId', 'childIds',
        'idx', 'index', 'pid', 'windowId', 'relation', 'relations', 'source', 'target',
        'bbox', 'rect', 'position', 'size', 'absolute_position', 'visible_bbox', 'clip',
        'x', 'y', 'width', 'height', 'top', 'left', 'bottom', 'right', 
        'color', 'backgroundColor', 'fgColor', 'bgColor', 'style', 'textStyle', 'layout',
        'is_pdf', 'superseded', 'busy', 'focusable', 'focused', 'editable', 
        'live', 'atomic', 'relevant', 'roledescription', 'level', 'posinset', 'setsize'
    }

    # Roles that imply "User Interface" even if empty.
    INTERACTIVE_ROLES = {
        'button', 'input', 'textfield', 'search', 'link', 'checkbox', 'switch', 
        'slider', 'menu', 'menuitem', 'combobox', 'listbox', 'tab', 'tree', 'table',
        'axbutton', 'axlink', 'aximage', 'axcheckbox', 'axtextfield', 
        'scrollbar', 'splitter', 'toolbar'
    }

    @staticmethod
    def load_ax(ax_path) -> list[dict]:
        with open(ax_path, "r", encoding="utf-8") as file:
            return json.load(file)

    @staticmethod
    def _is_garbage_text(s: str) -> bool:
        if len(s) < 2: return True 
        if s.lower() in {'true', 'false', 'none', 'null', 'undefined', 'div', 'span', 'body', 'view'}: return True
        if s.startswith('#') and len(s) < 8: return True 
        return False

    @staticmethod
    def harvest_strings(node: dict) -> tuple[str | None, list[str]]:
        role = None
        found_text = []
        stack = [node]
        
        while stack:
            curr = stack.pop()
            if not isinstance(curr, dict): continue

            for k, v in curr.items():
                if k in AXCompressor8.IGNORED_KEYS: continue
                
                # A. Capture Role
                if k.lower() in {'role', 'chromerole', 'axrole', 'role_description'}:
                    if not role or (isinstance(v, str) and len(v) < 20):
                        role = v
                    continue

                # B. Skip recursion keys
                if k in {'children', 'childIds', 'nodes', 'AXChildren'}:
                    continue

                # C. Capture Strings (The Vacuum)
                if isinstance(v, str):
                    s = v.strip()
                    if s and not AXCompressor8._is_garbage_text(s):
                        if k in {'url', 'href', 'docUrl'} and s.startswith('http'):
                            found_text.append(f"URL: {s}")
                        elif s not in found_text:
                            found_text.append(s)
                
                # D. Capture Numbers (sliders, values)
                elif isinstance(v, (int, float)) and not isinstance(v, bool):
                     if k.lower() in {'value', 'axvalue', 'valuenow'}:
                        found_text.append(str(v))

                # E. Recurse
                elif isinstance(v, dict):
                    stack.append(v)
                elif isinstance(v, list):
                    for item in v:
                        if isinstance(item, dict): stack.append(item)
                        elif isinstance(item, str) and not AXCompressor8._is_garbage_text(item):
                            if item not in found_text: found_text.append(item)

        return role, found_text

    @staticmethod
    def clean_and_compress(node: dict, is_root: bool = False) -> dict | None:
        # 1. Visibility
        if node.get('visible') is False: return None
        if node.get('hidden') is True: return None

        # 2. Harvest
        role, content_list = AXCompressor8.harvest_strings(node)
        
        clean_node = {}
        if role: clean_node['r'] = role
        if content_list:
            # Dedupe preserving order
            clean_node['d'] = " | ".join(dict.fromkeys(content_list))

        # 3. Children
        raw_children = (node.get('children') or node.get('childIds') or 
                        node.get('nodes') or node.get('AXChildren') or [])
        
        processed_children = []
        if isinstance(raw_children, list):
            for child in raw_children:
                if isinstance(child, dict):
                    res = AXCompressor8.clean_and_compress(child, is_root=False)
                    if res: processed_children.append(res)

        # 4. Dedupe Child Echoes
        parent_text = clean_node.get('d', "")
        final_children = []
        for child in processed_children:
            child_text = child.get('d', "")
            child_role = str(child.get('r', "")).lower()
            
            # If child text == parent text, and child is a leaf
            if child_text and parent_text and child_text == parent_text and 'ch' not in child:
                # If roles match (Text in Text) -> Delete child
                if child_role == str(role).lower(): continue
                # If child is generic -> Delete child
                if child_role in {'statictext', 'generic', 'text', 'group'}: continue
            
            final_children.append(child)
        processed_children = final_children

        # 5. Hoisting / Flattening
        has_content = 'd' in clean_node
        
        is_interactive = False
        if role:
            r_low = str(role).lower()
            is_interactive = any(x in r_low for x in AXCompressor8.INTERACTIVE_ROLES)

        # A. Dead Leaf
        if not has_content and not processed_children:
            if is_root or is_interactive: return clean_node
            return None

        # B. Useless Wrapper
        # If I am not Root, not Interactive, have no text, and exactly 1 child -> Become the child
        if not is_root and not is_interactive and not has_content and len(processed_children) == 1:
            return processed_children[0]

        if processed_children:
            clean_node['ch'] = processed_children

        return clean_node

    @staticmethod
    def to_string(node: dict, depth=0) -> str:
        indent = "  " * depth
        role = node.get('r', 'Group')
        desc = f' "{node["d"]}"' if 'd' in node else ""
        lines = [f"{indent}[{role}]{desc}"]
        for child in node.get('ch', []):
            lines.append(AXCompressor8.to_string(child, depth + 1))
        return "\n".join(lines)

    @staticmethod
    def process_ax(data: list[dict], output_format='json') -> list[dict]:
        final_output = []

        for i, row in enumerate(data):
            ax = row.get('accessibility', {})
            raw_root = ax.get('data')
            
            # --- GUARD: Empty Record handling ---
            if not raw_root:
                # Fallback record so index 'i' stays aligned
                final_output.append({'id': i, 'type': 'EMPTY', 'accessibility': '[Empty]'})
                continue
            
            # List Unwrap
            if isinstance(raw_root, list):
                if not raw_root:
                    final_output.append({'id': i, 'type': 'EMPTY', 'accessibility': '[Empty]'})
                    continue
                # If multiple windows, keep them
                if len(raw_root) > 1:
                    raw_root = {"role": "VirtualRoot", "children": raw_root}
                else:
                    raw_root = raw_root[0]

            if not isinstance(raw_root, dict):
                final_output.append({'id': i, 'type': 'ERROR', 'accessibility': '[Error]'})
                continue

            # --- PROCESS ---
            if raw_root.get('is_pdf'):
                summary = raw_root.get('pdf_summary', {})
                txt = str(summary.get('text', summary) if isinstance(summary, dict) else summary)
                clean_root = {"r": "PDF", "d": txt[:1500]}
                type_label = "PDF"
            else:
                clean_root = AXCompressor8.clean_and_compress(raw_root, is_root=True)
                type_label = 'BROWSER' if row.get('accessibility', {}).get('is_browser') else 'APP'
            
            # --- OUTPUT ---
            if clean_root:
                payload = AXCompressor8.to_string(clean_root) if output_format == 'string' else clean_root
            else:
                # If cleaning removed everything, output placeholder
                payload = "[Empty Accessibility Tree]"
            
            final_output.append({
                'id': i,
                'type': type_label,
                'accessibility': payload
            })

        return final_output

In [109]:
raw_data = AXCompressor8.load_ax(raw_path)
compressed_data = AXCompressor8.process_ax(raw_data, output_format='string')

# 2. Run the Validator
AXValidator.validate_dataset(raw_path, compressed_data)

ID    | Score  | Status     | Missing Words Sample
----------------------------------------------------------------------------------------------------
0     | 66.67% | ⚠️ LOW     | axgroup, axwindow, axbutton
1     | 66.67% | ⚠️ LOW     | axgroup, axwindow, axbutton
2     | 70.0% | ⚠️ LOW     | axgroup, axwindow, axbutton
4     | 99.95% | ✅ OK       | inlinetextbox
5     | 99.95% | ✅ OK       | inlinetextbox
6     | 99.95% | ✅ OK       | inlinetextbox
7     | 100.0% | ✅ OK       | 
8     | 100.0% | ✅ OK       | 
9     | 100.0% | ✅ OK       | 
10    | 100.0% | ✅ OK       | 
11    | 100.0% | ✅ OK       | 
12    | 70.0% | ⚠️ LOW     | axgroup, axwindow, axbutton
13    | 70.0% | ⚠️ LOW     | axgroup, axwindow, axbutton
14    | 72.73% | ⚠️ LOW     | axgroup, axwindow, axbutton
15    | 99.44% | ✅ OK       | inlinetextbox
16    | 99.44% | ✅ OK       | inlinetextbox
17    | 99.43% | ✅ OK       | inlinetextbox
18    | 43.71% | ❌ BAD      | researches, way, directions, works, models, axrow, age

In [115]:
print(compressed_data[340]['accessibility'])

[VirtualRoot]
  [window] "Finder"
  [standard window] "Downloads | Finder"
    [split group]
      [outline] "sidebar"
        [text] "All Tags…"
        [text] "Home"
        [text] "Work"
        [text] "Purple"
        [text] "Grey"
        [text] "Blue"
        [text] "Green"
        [text] "Yellow"
        [text] "Orange"
        [text] "Tags"
        [cell]
          [image] "bin"
          [text] "Bin"
        [text] "AirDrop"
        [cell]
          [button] "Eject | eject"
          [text] "Untitled"
        [cell]
          [image] "home"
          [text] "monsteroideae"
        [cell]
          [image] "move"
          [text] "Google Drive"
        [cell]
          [image] "iCloud"
          [text] "iCloud Drive"
        [text] "Locations"
        [text] "Desktop"
        [cell]
          [image] "Arrow Down Circle"
          [text] "Downloads"
        [cell]
          [image] "move"
          [text] "Research"
        [cell]
          [image] "document"
          [text] "D

In [98]:
class AXValidatorStrict:
    """
    A smarter validator that:
    1. Ignores content from nodes marked 'visible: false' in Raw data.
    2. Ignores hash-like / base64-like strings.
    3. Ignores irrelevant keys (id, class, etc).
    """

    # We only care if we lose text from these specific semantic keys
    SEMANTIC_KEYS = {'value', 'name', 'label', 'title', 'description', 'text', 'hint', 'role', 'url', 'href'}
    
    # Regex to catch "Real Words" vs "Garbage"
    # Good: "Submit", "login", "cancel"
    # Bad: "a8f9", "x-23", "fzr3v636c8g"
    GARBAGE_REGEX = re.compile(r'^(?:[0-9]+|true|false|null|undefined|[a-z0-9]{1,3}|[a-z0-9]{12,}|0x[0-9a-f]+)$', re.IGNORECASE)
    
    @staticmethod
    def is_visible(node):
        """Returns False only if explicitly hidden."""
        if isinstance(node, dict):
            if node.get('visible') is False: return False
            if node.get('hidden') is True: return False
        return True

    @staticmethod
    def tokenize(text: str) -> set[str]:
        if not text: return set()
        # Split by non-alphanumeric
        tokens = re.split(r'[^a-zA-Z0-9]+', str(text).lower())
        # Filter garbage
        valid_tokens = {t for t in tokens if t and not AXValidatorStrict.GARBAGE_REGEX.match(t)}
        return valid_tokens

    @staticmethod
    def extract_tokens_from_raw(node: dict | list) -> set[str]:
        tokens = set()
        
        if isinstance(node, list):
            for item in node:
                tokens.update(AXValidatorStrict.extract_tokens_from_raw(item))
            return tokens
        
        if isinstance(node, dict):
            # CRITICAL FIX: If node is invisible, ignore its content
            if not AXValidatorStrict.is_visible(node):
                return set()

            for k, v in node.items():
                # Only look at semantic keys
                if k in AXValidatorStrict.SEMANTIC_KEYS and isinstance(v, (str, int)):
                    tokens.update(AXValidatorStrict.tokenize(v))
                
                # Recurse into attributes/children
                if isinstance(v, (dict, list)):
                    tokens.update(AXValidatorStrict.extract_tokens_from_raw(v))
        return tokens

    @staticmethod
    def extract_tokens_from_compressed(compressed_data) -> set[str]:
        if isinstance(compressed_data, str):
            return AXValidatorStrict.tokenize(compressed_data)
        elif isinstance(compressed_data, (dict, list)):
            return AXValidatorStrict.extract_tokens_from_raw(compressed_data)
        return set()

    @staticmethod
    def validate(raw_data, compressed_output):
        print(f"{'ID':<5} | {'Score':<6} | {'Status':<10} | {'REAL Missing Content (No Garbage)'}")
        print("-" * 100)

        scores = []
        for i, raw_row in enumerate(raw_data):
            if i >= len(compressed_output): break
            
            comp_row = compressed_output[i]
            
            # 1. Get Roots
            raw_root = raw_row.get('accessibility', {}).get('data')
            comp_payload = comp_row.get('accessibility')
            
            if not raw_root: continue

            # 2. Extract
            raw_tokens = AXValidatorStrict.extract_tokens_from_raw(raw_root)
            comp_tokens = AXValidatorStrict.extract_tokens_from_compressed(comp_payload)

            # 3. Compare
            missing = raw_tokens - comp_tokens
            
            # Calculate Score
            if len(raw_tokens) == 0:
                score = 100.0
            else:
                score = 100.0 * (1 - (len(missing) / len(raw_tokens)))
            
            scores.append(score)

            # 4. Filter Output
            if score < 90:
                status = "✅ OK" if score > 80 else "❌ BAD"
                missing_str = ", ".join(list(missing)[:8])
                r_id = str(raw_row.get('id', i))
                print(f"{r_id:<5} | {round(score,1)}%  | {status:<10} | {missing_str}")

        print("-" * 100)
        print(f"Average: {sum(scores)/len(scores):.2f}%")

In [99]:
class AXCompressor6:
    """
    The 'No-Loss' version. 
    1. Deep searches attributes.
    2. Explicitly prioritizes merging Value + Name + Description.
    3. Handles standard browser URL patterns.
    """

    # We strip these, but process everything else.
    IGNORED_KEYS = {
        'id', 'nodeId', 'backendDOMNodeId', 'parentId', 'frameId', 'childIds', 'index',
        'bbox', 'rect', 'position', 'size', 'absolute_position', 'visible_bbox', 
        'x', 'y', 'width', 'height', 'top', 'left', 'bottom', 'right', 'clip',
        'color', 'backgroundColor', 'style', 'layout', 'is_pdf', 'superseded'
    }

    @staticmethod
    def load_ax(ax_path) -> list[dict]:
        with open(ax_path, "r", encoding="utf-8") as file:
            return json.load(file)

    @staticmethod
    def _clean_str(s):
        """Cleans whitespace and checks for garbage."""
        if not isinstance(s, str): return ""
        s = s.strip()
        if len(s) < 2 and s not in {'x', 'y'}: return "" # Skip single chars unless math
        if s.lower() in {'true', 'false', 'none', 'null', 'undefined'}: return ""
        return s

    @staticmethod
    def extract_all_text(node: dict) -> tuple[str, list[str]]:
        role = node.get('role') or node.get('chromeRole') or node.get('role_description')
        
        # We use a set to avoid duplicates like Name="Login", Value="Login"
        found_text = set()

        def recurse_find_text(d):
            if not isinstance(d, dict): return
            
            for k, v in d.items():
                if k in AXCompressor6.IGNORED_KEYS: continue
                
                # If specific known keys, prioritize them
                if k in {'value', 'name', 'label', 'description', 'title', 'text', 'caption'}:
                    if isinstance(v, str):
                        clean = AXCompressor6._clean_str(v)
                        if clean: found_text.add(clean)
                    elif isinstance(v, (int, float)) and not isinstance(v, bool):
                        found_text.add(str(v))
                
                # If URL/HREF (Crucial for browser)
                elif k in {'url', 'href', 'src'}:
                    if isinstance(v, str) and len(v) > 5:
                        found_text.add(f"url:{v}")

                # Recurse into attributes/properties
                elif isinstance(v, dict):
                    recurse_find_text(v)
                elif isinstance(v, list):
                    for i in v: 
                        if isinstance(i, dict): recurse_find_text(i)
                        elif isinstance(i, str): 
                             clean = AXCompressor6._clean_str(i)
                             if clean: found_text.add(clean)

        recurse_find_text(node)
        
        # Sort so output is deterministic
        return role, sorted(list(found_text))

    @staticmethod
    def clean_and_compress(node: dict, is_root: bool = False) -> dict | None:
        # 1. Visibility
        if node.get('visible') is False: return None
        if node.get('hidden') is True: return None

        # 2. Extract Content
        role, content_list = AXCompressor6.extract_all_text(node)
        
        clean_node = {}
        if role: clean_node['r'] = role
        if content_list: clean_node['d'] = " | ".join(content_list)

        # 3. Children
        raw_children = (node.get('children') or node.get('childIds') or node.get('nodes') or [])
        processed_children = []
        
        if isinstance(raw_children, list):
            for child in raw_children:
                if isinstance(child, dict):
                    res = AXCompressor6.clean_and_compress(child, is_root=False)
                    if res: processed_children.append(res)

        # 4. Deduplication (Echo Removal)
        parent_txt = clean_node.get('d', "")
        final_children = []
        for child in processed_children:
            child_txt = child.get('d', "")
            # If child has same text as parent and no own children, it's redundant
            if child_txt and child_txt == parent_txt and 'ch' not in child:
                continue
            final_children.append(child)
        processed_children = final_children

        # 5. Hoisting (Flattening)
        has_content = 'd' in clean_node
        
        # Dead Leaf
        if not has_content and not processed_children:
            return clean_node if is_root else None

        # Useless Wrapper (Hoist child up)
        # Don't hoist if Root, or if Role is interactive (Button)
        is_interactive = role and any(x in str(role).lower() for x in ['button', 'input', 'link', 'check'])
        
        if not is_root and not has_content and len(processed_children) == 1 and not is_interactive:
            return processed_children[0]

        if processed_children:
            clean_node['ch'] = processed_children

        return clean_node

    @staticmethod
    def to_string(node: dict, depth=0) -> str:
        indent = "  " * depth
        role = node.get('r', 'Group')
        desc = f' "{node["d"]}"' if 'd' in node else ""
        lines = [f"{indent}[{role}]{desc}"]
        for child in node.get('ch', []):
            lines.append(AXCompressor6.to_string(child, depth + 1))
        return "\n".join(lines)

    @staticmethod
    def process_ax(data: list[dict], output_format='json') -> list[dict]:
        final_output = []
        for i, row in enumerate(data):
            ax = row.get('accessibility', {})
            raw_root = ax.get('data')
            if not raw_root: continue
            
            # Unwrap list if needed
            if isinstance(raw_root, list):
                if not raw_root: continue
                raw_root = raw_root[0] # Simplification for context

            if isinstance(raw_root, dict):
                if raw_root.get('is_pdf'):
                    # PDF
                    txt = str(raw_root.get('pdf_summary', {}))
                    clean_root = {"r": "PDF", "d": txt[:1000]}
                else:
                    # APP / BROWSER
                    clean_root = AXCompressor6.clean_and_compress(raw_root, is_root=True)
                
                if clean_root:
                    payload = AXCompressor6.to_string(clean_root) if output_format == 'string' else clean_root
                    final_output.append({
                        'id': i, 
                        'type': 'AX', 
                        'accessibility': payload
                    })
        return final_output

In [102]:
class AXCompressor7:
    """
    The 'Harvester' Version.
    1. Iterates EVERY key in the node.
    2. Keeps ANY string that isn't on the Garbage Blocklist.
    3. Never deletes interactive roles (Buttons, Inputs) even if empty.
    4. recursively flattens attributes to find hidden text.
    """

    # --- 1. THE GARBAGE BLOCKLIST ---
    # We ignore these keys completely. EVERYTHING ELSE IS SCANNED.
    IGNORED_KEYS = {
        # Tech IDs & References
        'id', 'nodeId', 'backendDOMNodeId', 'backendNodeId', 'parentId', 'frameId', 'childIds',
        'idx', 'index', 'pid', 'windowId', 'relation', 'relations', 'source', 'target',
        # Geometry & Visual Layout (The biggest token wasters)
        'bbox', 'rect', 'position', 'size', 'absolute_position', 'visible_bbox', 'clip',
        'x', 'y', 'width', 'height', 'top', 'left', 'bottom', 'right', 
        'color', 'backgroundColor', 'fgColor', 'bgColor', 'style', 'textStyle', 'layout',
        # Meta flags
        'is_pdf', 'superseded', 'busy', 'focusable', 'focused', 'editable', 
        'live', 'atomic', 'relevant', 'roledescription' 
    }

    # Roles that MUST exist in the tree, even if they have no text.
    # (e.g. an empty text field is still an input).
    INTERACTIVE_ROLES = {
        'button', 'input', 'textfield', 'search', 'link', 'checkbox', 'switch', 
        'slider', 'menu', 'menuitem', 'combobox', 'listbox', 'tab', 'tree',
        'axbutton', 'axlink', 'aximage', 'axcheckbox', 'axtextfield'
    }

    @staticmethod
    def load_ax(ax_path) -> list[dict]:
        with open(ax_path, "r", encoding="utf-8") as file:
            return json.load(file)

    @staticmethod
    def _is_garbage_text(s: str) -> bool:
        """Returns True if string is likely tech noise."""
        if len(s) < 2: return True # Single chars (unless customized)
        if s.lower() in {'true', 'false', 'none', 'null', 'undefined', 'div', 'span', 'body'}: return True
        # Check for hex colors or weird IDs
        if s.startswith('#') and len(s) < 8: return True 
        return False

    @staticmethod
    def harvest_strings(node: dict) -> tuple[str | None, list[str]]:
        """
        Recursively scans the dict to find the Role and ANY other text.
        """
        role = None
        found_text = []
        
        # We use a stack to process nested dicts (like 'attributes') without deep recursion limits
        stack = [node]
        
        while stack:
            curr = stack.pop()
            if not isinstance(curr, dict): continue

            for k, v in curr.items():
                if k in AXCompressor7.IGNORED_KEYS: continue
                
                # A. Handle Role
                if k.lower() in {'role', 'chromerole', 'axrole', 'role_description'}:
                    # We prefer the most specific role we find
                    if not role or (isinstance(v, str) and len(v) < 20):
                        role = v
                    continue

                # B. Handle Children/Nodes (Don't harvest text from them yet, just skip)
                if k in {'children', 'childIds', 'nodes', 'AXChildren'}:
                    continue

                # C. Handle String Values
                if isinstance(v, str):
                    s = v.strip()
                    if s and not AXCompressor7._is_garbage_text(s):
                        # Special handling for URLs to make them cleaner
                        if k in {'url', 'href', 'docUrl'} and s.startswith('http'):
                            found_text.append(f"URL: {s}")
                        elif s not in found_text:
                            found_text.append(s)
                
                # D. Handle Numbers (only if meaningful, e.g. slider value)
                elif isinstance(v, (int, float)) and not isinstance(v, bool):
                    # Filter out likely coordinates (integers > 1900 might be years, but often coords)
                    # This is a heuristic.
                    if k in {'value', 'axvalue', 'level', 'posinset'}:
                        found_text.append(str(v))

                # E. Recurse into attributes/properties
                elif isinstance(v, dict):
                    stack.append(v)
                elif isinstance(v, list):
                    for item in v:
                        if isinstance(item, dict): stack.append(item)
                        elif isinstance(item, str) and not AXCompressor7._is_garbage_text(item):
                            if item not in found_text: found_text.append(item)

        return role, found_text

    @staticmethod
    def clean_and_compress(node: dict, is_root: bool = False) -> dict | None:
        # 1. VISIBILITY CHECK
        # If explicitly hidden, kill it.
        if node.get('visible') is False: return None
        if node.get('hidden') is True: return None
        # Browser nodes sometimes use 'box' or 'rect' = [0,0,0,0] to denote invisible
        # but checking explicit flags is safer.

        # 2. HARVEST CONTENT
        role, content_list = AXCompressor7.harvest_strings(node)
        
        clean_node = {}
        if role: 
            clean_node['r'] = role
        
        # Deduplicate content list and join
        if content_list:
            # OrderedDict trick to dedupe while preserving order
            unique_content = list(dict.fromkeys(content_list))
            clean_node['d'] = " | ".join(unique_content)

        # 3. PROCESS CHILDREN
        raw_children = (node.get('children') or node.get('childIds') or 
                        node.get('nodes') or node.get('AXChildren') or [])
        
        processed_children = []
        if isinstance(raw_children, list):
            for child in raw_children:
                if isinstance(child, dict):
                    res = AXCompressor7.clean_and_compress(child, is_root=False)
                    if res: processed_children.append(res)

        # 4. PARENT-CHILD DEDUPLICATION (The "Echo" Fix)
        # Only delete child if:
        #   1. Child text == Parent text
        #   2. Child has NO children of its own (is a leaf)
        #   3. Child role is generic (Group/StaticText) OR matches parent role
        #   (We don't want to delete a Button inside a Group just because they have same text)
        
        parent_text = clean_node.get('d', "")
        parent_role = str(clean_node.get('r', "")).lower()
        
        final_children = []
        for child in processed_children:
            child_text = child.get('d', "")
            child_role = str(child.get('r', "")).lower()
            
            is_echo = (child_text and parent_text and child_text == parent_text)
            is_leaf = 'ch' not in child
            
            # Safe to delete?
            if is_echo and is_leaf:
                # If roles are identical, it's definitely redundant (Text inside Text)
                if child_role == parent_role:
                    continue
                # If child is generic text inside a semantic parent (Text inside Button), drop text
                if 'statictext' in child_role or 'generic' in child_role:
                    continue
            
            final_children.append(child)
        
        processed_children = final_children

        # 5. HOISTING / FLATTENING (The "Wrapper" Fix)
        has_content = 'd' in clean_node
        
        # Check if "Interactive" (Must keep)
        is_interactive = False
        if role:
            r_low = str(role).lower()
            is_interactive = any(x in r_low for x in AXCompressor7.INTERACTIVE_ROLES)

        # A. DEAD LEAF
        # No content, no children.
        if not has_content and not processed_children:
            # KEEP if Root or Interactive. DELETE otherwise.
            if is_root or is_interactive: 
                return clean_node
            return None

        # B. USELESS WRAPPER
        # No content, 1 child.
        if not is_root and not has_content and len(processed_children) == 1:
            # NEVER hoist if self is Interactive (e.g. A Button wrapping an Icon)
            if not is_interactive:
                return processed_children[0]

        # Attach children
        if processed_children:
            clean_node['ch'] = processed_children

        return clean_node

    @staticmethod
    def to_string(node: dict, depth=0) -> str:
        indent = "  " * depth
        role = node.get('r', 'Group')
        desc = f' "{node["d"]}"' if 'd' in node else ""
        
        # Output format: [Role] "Description"
        line = f"{indent}[{role}]{desc}"
        
        lines = [line]
        for child in node.get('ch', []):
            lines.append(AXCompressor7.to_string(child, depth + 1))
        return "\n".join(lines)

    @staticmethod
    def process_ax(data: list[dict], output_format='json') -> list[dict]:
        final_output = []

        for i, row in enumerate(data):
            ax = row.get('accessibility', {})
            raw_root = ax.get('data')
            if not raw_root: continue
            
            # Normalize List/Dict Root
            if isinstance(raw_root, list):
                if not raw_root: continue
                # Strategy: If multiple windows, create a Virtual Root
                if len(raw_root) > 1:
                    raw_root = {"role": "VirtualRoot", "children": raw_root}
                else:
                    raw_root = raw_root[0]

            if not isinstance(raw_root, dict): continue

            # PDF Handling (Special case)
            if raw_root.get('is_pdf'):
                summary = raw_root.get('pdf_summary', {})
                txt = str(summary.get('text', summary) if isinstance(summary, dict) else summary)
                clean_root = {"r": "PDF", "d": txt[:1500]} # Cap PDF length
                type_label = "PDF"
            else:
                # MAIN PROCESSING
                clean_root = AXCompressor7.clean_and_compress(raw_root, is_root=True)
                type_label = 'BROWSER' if row.get('accessibility', {}).get('is_browser') else 'APP'
            
            if clean_root:
                payload = AXCompressor7.to_string(clean_root) if output_format == 'string' else clean_root
                final_output.append({
                    'id': i,
                    'type': type_label,
                    'accessibility': payload
                })

        return final_output

In [110]:
raw_data = AXCompressor8.load_ax(raw_path)
compressed_data = AXCompressor8.process_ax(raw_data, output_format='string')

# 2. Run the Validator
AXValidatorStrict.validate(raw_data, compressed_data)

ID    | Score  | Status     | REAL Missing Content (No Garbage)
----------------------------------------------------------------------------------------------------
0     | 57.1%  | ❌ BAD      | axgroup, axwindow, axbutton
1     | 57.1%  | ❌ BAD      | axgroup, axwindow, axbutton
2     | 62.5%  | ❌ BAD      | axgroup, axwindow, axbutton
12    | 62.5%  | ❌ BAD      | axgroup, axwindow, axbutton
13    | 62.5%  | ❌ BAD      | axgroup, axwindow, axbutton
14    | 66.7%  | ❌ BAD      | axgroup, axwindow, axbutton
18    | 83.6%  | ✅ OK       | axgroup, axtextfield, axtoolbar, axrow, axsplitter, axcolumn, axtable, axtextarea
19    | 81.2%  | ✅ OK       | axgroup, axtextfield, axtoolbar, axrow, axsplitter, axcolumn, axtable, axbutton
108   | 66.7%  | ❌ BAD      | axgroup, axwindow, axbutton
109   | 62.5%  | ❌ BAD      | axgroup, axwindow, axbutton
110   | 62.5%  | ❌ BAD      | axgroup, axwindow, axbutton
142   | 25.0%  | ❌ BAD      | axgroup, axwindow, axbutton
143   | 40.0%  | ❌ BAD      | axg

In [112]:
len(str(compressed_data).split(' '))

2068422

In [117]:
len(compressed_data)

350

In [118]:
[len(str(rec['accessibility']).split(' ')) for rec in compressed_data]

[18,
 18,
 18,
 1,
 69174,
 69174,
 69174,
 45,
 45,
 45,
 45,
 45,
 18,
 18,
 18,
 3233,
 3233,
 3233,
 1415,
 1360,
 1,
 3233,
 45,
 45,
 45,
 3233,
 3233,
 45,
 346,
 346,
 346,
 346,
 346,
 346,
 346,
 3233,
 45,
 45,
 45,
 45,
 346,
 346,
 346,
 45,
 45,
 3233,
 3233,
 3233,
 3233,
 3233,
 346,
 346,
 8818,
 8818,
 7858,
 12,
 12,
 12,
 3233,
 3233,
 12,
 12,
 45,
 44524,
 44524,
 3233,
 3233,
 3233,
 3233,
 3233,
 3233,
 3233,
 45,
 45,
 45,
 45,
 45,
 45,
 45,
 45,
 45,
 45,
 45,
 177,
 4443,
 4449,
 4443,
 4443,
 3233,
 3233,
 3018,
 2360,
 2332,
 2332,
 2332,
 2332,
 2332,
 2378,
 2378,
 2378,
 3232,
 3232,
 3232,
 3232,
 3232,
 3232,
 45,
 45,
 18,
 18,
 18,
 26113,
 3232,
 3232,
 3232,
 3232,
 45,
 45,
 45,
 45,
 45,
 45,
 45,
 7009,
 157,
 6781,
 6787,
 6787,
 6787,
 3232,
 3232,
 3232,
 3232,
 3226,
 119,
 119,
 45,
 45,
 18049,
 3767,
 3767,
 3767,
 13,
 18,
 18,
 18,
 12,
 18039,
 12,
 12,
 12,
 45,
 45,
 45,
 119,
 119,
 119,
 119,
 119,
 22220,
 22220,
 22220,
 16820,


In [122]:
print(compressed_data[4]['accessibility'])

[Group]
  [{'type': 'internalRole', 'value': 'RootWebArea'}] "url | string | https://huggingface.co/datasets/xlangai/AgentNet | busy | boolean | 1 | focusable | booleanOrUndefined | computedString | xlangai/AgentNet · Datasets at Hugging Face | relatedElement | title | attribute | aria-label | aria-labelledby"
  [{'type': 'role', 'value': 'none'}] "uninteresting | boolean"
  [{'type': 'role', 'value': 'none'}] "uninteresting | boolean"
  [{'type': 'role', 'value': 'none'}] "uninteresting | boolean"
  [{'type': 'role', 'value': 'banner'}] "computedString | attribute | title | aria-label | relatedElement | aria-labelledby"
  [{'type': 'role', 'value': 'main'}] "computedString | attribute | title | aria-label | relatedElement | aria-labelledby"
  [{'type': 'role', 'value': 'contentinfo'}] "computedString | attribute | title | aria-label | relatedElement | aria-labelledby"
  [{'type': 'role', 'value': 'none'}] "uninteresting | boolean"
  [{'type': 'role', 'value': 'none'}] "uninteresting |

# The AX №8 does a good job